In [1]:
from striprtf.striprtf import rtf_to_text
import pandas as pd
import os
import re


os.chdir("/Users/Yannick/Documents/PhD/text_mining/geothermal")


def load_all_rtf_articles(directory):
    all_articles = []

    # Loop through all .rtf files
    for filename in os.listdir(directory):
        if not filename.lower().endswith(".rtf"):
            continue  # skip non-RTF files

        file_path = os.path.join(directory, filename)

        # Read file
        with open(file_path, "r", encoding="utf-8") as file:
            rtf_content = file.read()

        # Convert RTF → text
        plain_text = rtf_to_text(rtf_content)

        # Split into articles
        articles = plain_text.split("End of Document")

        # Extract data from each article
        for article in articles:
            if not article.strip():
                continue

            title = re.search(r'^(.*?)\n', article)
            newspaper = re.search(r'\n(.*?)\n\d{1,2} .*? \d{4}', article)
            date = re.search(r'\n(\d{1,2} .*? \d{4})', article)
            section = re.search(r'Section:\s*(.*?);', article)
            word_count = re.search(r'Length:\s*(\d+)', article)
            load_date = re.search(r'Load-Date:\s*(.*)', article)
            body = re.search(
                r'Body\n\n(.*?)\n\n(?:Load-Date:|Copyright|End of Document)',
                article,
                re.DOTALL
            )

            all_articles.append({
                'source_file': filename,
                'title': title.group(1).strip() if title else '',
                'newspaper': newspaper.group(1).strip() if newspaper else '',
                'date': date.group(1).strip() if date else '',
                'section': section.group(1).strip() if section else '',
                'word_count': int(word_count.group(1)) if word_count else None,
                'body': body.group(1).strip() if body else '',
                'load_date': load_date.group(1).strip() if load_date else ''
            })

    # Convert to DataFrame
    return pd.DataFrame(all_articles)


# ---- Run the function ----
df = load_all_rtf_articles("input_data")

# Inspect the first rows
print(df.head(10))
print(f"\nLoaded {len(df)} articles from folder.")


               source_file title                      newspaper  \
0  Bestanden (500) (3).RTF                         Nieuwe Oogst   
1  Bestanden (500) (3).RTF                         Nieuwe Oogst   
2  Bestanden (500) (3).RTF                      De Gelderlander   
3  Bestanden (500) (3).RTF                                Trouw   
4  Bestanden (500) (3).RTF                             Trouw.nl   
5  Bestanden (500) (3).RTF                  AD/Haagsche Courant   
6  Bestanden (500) (3).RTF                  AD/Haagsche Courant   
7  Bestanden (500) (3).RTF                Noordhollands Dagblad   
8  Bestanden (500) (3).RTF        De Twentsche Courant Tubantia   
9  Bestanden (500) (3).RTF                        De Stentor.nl   

               date           section  word_count  \
0  24 augustus 2019                           483   
1  24 augustus 2019                           123   
2  26 augustus 2019   De Gelderlander         293   
3  26 augustus 2019           Vandaag         832 

In [6]:
import geopandas as gpd

regions_gdf = gpd.read_file("output/other/regions_gdf_2025.geojson")

NATIONAL_KEY = "NL_NATIONAL"   # must exist in regions_gdf["newspaper"]

import geopandas as gpd

if regions_gdf.crs is None:
    regions_gdf = regions_gdf.set_crs("EPSG:4326")
else:
    regions_gdf = regions_gdf.to_crs("EPSG:4326")

if NATIONAL_KEY not in set(regions_gdf["newspaper"]):
    # Create NL geometry as union of all regions (works because you already have province/municipality polygons)
    nl_geom = regions_gdf.geometry.union_all()

    regions_gdf = pd.concat(
        [
            regions_gdf,
            gpd.GeoDataFrame(
                [{
                    "newspaper": NATIONAL_KEY,
                    "location_classification": "NL",
                    "region_name": "Nederland",
                    "geometry": nl_geom
                }],
                crs="EPSG:4326"
            )
        ],
        ignore_index=True
    )


In [7]:
# -------------------------------------------------------
# 1. FILTER OUT MISSING NEWSPAPER + MISSING DATE
# -------------------------------------------------------

missing_newspaper = df[df['newspaper'].isna() | (df['newspaper'].str.strip() == "")]
print("Number of missing newspapers:", len(missing_newspaper))

missing_dates = df[df['date'].isna() | (df['date'].str.strip() == "")]
print("Number of missing dates:", len(missing_dates))

# Remove rows missing newspaper or date
df_cleaned = df[
    ~(df['newspaper'].isna() | (df['newspaper'].str.strip() == ""))
].copy()

df_cleaned = df_cleaned[
    ~(df_cleaned['date'].isna() | (df_cleaned['date'].str.strip() == ""))
].copy()

df_cleaned = df_cleaned.reset_index(drop=True)

print("Rows after filtering missing data:", len(df_cleaned))

# -------------------------------------------------------
# 2. DUTCH → ENGLISH MONTH CLEANING + DATE PARSING
# -------------------------------------------------------

month_map = {
    "januari": "January",
    "februari": "February",
    "maart": "March",
    "april": "April",
    "mei": "May",
    "juni": "June",
    "juli": "July",
    "augustus": "August",
    "september": "September",
    "oktober": "October",
    "november": "November",
    "december": "December"
}

def dutch_to_english_month(date_str: str) -> str:
    if pd.isna(date_str):
        return date_str
    s = str(date_str).lower()
    for nl, en in month_map.items():
        s = re.sub(rf"\b{nl}\b", en, s)
    return s

df_cleaned["date_str_clean"] = df_cleaned["date"].astype(str).apply(dutch_to_english_month)

df_cleaned["date_dt"] = pd.to_datetime(
    df_cleaned["date_str_clean"],
    dayfirst=True,
    errors="coerce"
)

print("Total rows:", len(df_cleaned))
print("NaT after parsing:", df_cleaned["date_dt"].isna().sum())

df_cleaned["date"] = df_cleaned["date_dt"]

# -------------------------------------------------------
# 3. DEDUPLICATE ON 'body' & COLLAPSE OTHER COLUMNS TO SETS
# -------------------------------------------------------

# Identify all columns except body
other_cols = [c for c in df_cleaned.columns if c != "body"]

def collapse_dates(series):
    """Return earliest parsed timestamp."""
    cleaned = series.dropna()
    if len(cleaned) == 0:
        return pd.NaT
    return cleaned.min()

def collapse_to_set(series):
    """Return unique values as a set."""
    return set(series.dropna().unique())

agg_dict = {}
for col in other_cols:
    if col == "date":
        agg_dict[col] = collapse_dates      # KEEP EARLIEST DATE
    else:
        agg_dict[col] = collapse_to_set     # EVERYTHING ELSE → set()

df_cleaned = (
    df_cleaned
    .groupby("body", as_index=False)
    .agg(agg_dict)
)

print("Rows after deduplicating on body:", len(df_cleaned))

# -------------------------------------------------------
# 4. WORD COUNT COLUMN (using 'body')
# -------------------------------------------------------

df_cleaned["word_count"] = (
    df_cleaned["body"]
    .astype(str)
    .str.split()
    .str.len()
)

# Inspect word count stats if needed
#print("\nWord count stats:")
#print(df_cleaned["word_count"].describe())


# -------------------------------------------------------
# 5. GEOTHERMAL KEYWORD DETECTION
# -------------------------------------------------------

geo_keywords = [
    r"aardwarmte\w*",     # aardwarmte*
    r"geotherm\w*",       # geotherm*
]

geo_pattern = r"\b(" + "|".join(geo_keywords) + r")\b"

df_cleaned["geo_hits"] = (
    df_cleaned["body"]
    .astype(str)
    .str.lower()
    .str.count(geo_pattern)
)

# Inspect geothermal hits distribution if needed
#print("\nDistribution of geothermal keyword hits:")
#print(df_cleaned["geo_hits"].value_counts().sort_index())

# -------------------------------------------------------
# 6. FILTERING — ADJUST THESE TWO THRESHOLDS
# -------------------------------------------------------

MIN_WORDS = 100      # Recommended starting value — inspect and adjust
MIN_GEO_HITS = 3     # Recommended starting value — stricter = more relevant corpus

df_final = df_cleaned[
    (df_cleaned["word_count"] >= MIN_WORDS) &
    (df_cleaned["geo_hits"] >= MIN_GEO_HITS)
].reset_index(drop=True)

print("\nNumber of documents after relevance filtering:", len(df_final))


# -------------------------------------------------------
# 7. FINAL DUPLICATE / MISSING CHECKS
# -------------------------------------------------------

df_final_hashable = df_final.copy()

for col in df_final_hashable.columns:
    df_final_hashable[col] = df_final_hashable[col].apply(
        lambda x: frozenset(x) if isinstance(x, set) else x
    )

duplicate_rows = df_final_hashable[df_final_hashable.duplicated()]
print("Full-row duplicates:", len(duplicate_rows))

missing_newspaper = df_final[df_final['newspaper'].isna() | (df_final['newspaper'].str.strip() == "")]
print("Missing newspapers after all cleaning:", len(missing_newspaper))

print("\nFINAL DOCUMENT COUNT:", len(df_final))

# -------------------------------------------------------
# 8. LANGUAGE DETECTION
# -------------------------------------------------------

from langdetect import detect
from langdetect.lang_detect_exception import LangDetectException

# Function to safely detect language
def detect_language(text):
    try:
        return detect(text)
    except LangDetectException:
        return "unknown"

# Apply to the body column (you can also use 'title' or combine both)
df_final['language'] = df_final['body'].apply(detect_language)

# Count language occurrences
print("Percentage Dutch documents:", df_final[['language']].value_counts()/len(df_final) * 100)

print("\nSample of final dataset:")
print(df_final.head(5))


Number of missing newspapers: 142
Number of missing dates: 142
Rows after filtering missing data: 3358
Total rows: 3358
NaT after parsing: 0
Rows after deduplicating on body: 2345

Number of documents after relevance filtering: 944
Full-row duplicates: 0
Missing newspapers after all cleaning: 0

FINAL DOCUMENT COUNT: 944
Percentage Dutch documents: language
nl          100.0
Name: count, dtype: float64

Sample of final dataset:
                                                body  \
0  "Door dit faillissement is het project af van ...   
1  'Als dit project nu niet lukt, moeten we op zo...   
2  'Antwoorden ontbreken nog steeds, we kiezen nu...   
3  'Coating van leidingen niet afdoende'\nAbdel I...   
4  'Dit is een postzegel binnen de bebouwde kom v...   

                  source_file title              newspaper       date  \
0  {Bestanden (501-1000).RTF}    {}               {Cobouw} 2013-07-30   
1   {Bestanden (500) (1).RTF}    {}   {Leeuwarder Courant} 2021-10-16   
2     {Besta

In [9]:
import spacy
import re
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# -----------------------------
# 1. Paragraph splitter
# -----------------------------
def get_paragraphs(text: str):
    """Split an article into paragraphs based on blank lines."""
    if text is None:
        return []
    # Split on one or more blank lines
    paras = re.split(r"\n\s*\n+", str(text))
    # Strip and drop empties
    paras = [p.strip() for p in paras if p.strip()]
    # Fallback: if nothing found, treat whole text as one paragraph
    return paras or [str(text).strip()]


# -----------------------------
# 2. Sentiment model (multilingual, works with many transformers versions)
# -----------------------------
model_name = "DTAI-KULeuven/robbert-v2-dutch-sentiment"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Index 0..4 -> star rating 1..5
def sentiment_of(text: str):
    """Return (coarse_label, stars, score) for a given text."""
    encoded = tokenizer(text, return_tensors="pt", truncation=True)
    with torch.no_grad():
        output = model(**encoded)
    scores = torch.softmax(output.logits, dim=1).numpy()[0]

    star_idx = int(np.argmax(scores))         # 0..4
    stars = star_idx + 1                      # 1..5
    score = float(np.max(scores))             # confidence

    if stars <= 1:
        label = "negative"
    elif stars == 2:
        label = "neutral"
    else:
        label = "positive"

    return label, stars, score


# -----------------------------
# 3. Build paragraph-level dataframe with ALL metadata
# -----------------------------
rows = []

# df_final is assumed to exist and contain:
# - 'body' (article text)
# - any metadata columns: 'newspaper', 'date', 'title', etc.
for idx, row in df_final.iterrows():
    article_text = row["body"]
    paragraphs = get_paragraphs(article_text)

    for p_idx, para in enumerate(paragraphs):
        # Optional: skip very short paragraphs
        if len(para.split()) < 5:
            continue

        label, stars, score = sentiment_of(para)

        # Start with all article metadata
        meta = row.to_dict()
        # (If you don't want to duplicate full body text in this table, you can drop it:)
        # meta.pop("body", None)

        meta.update({
            "article_index": idx,          # index in df_cleaned
            "paragraph_id": p_idx,         # paragraph number within the article
            "paragraph_text": para,        # the paragraph text
            "sentiment": label,            # negative / neutral / positive
            "sentiment_stars": stars,      # 1..5 star rating
            "sentiment_score": score,      # model confidence
        })

        rows.append(meta)

df_paragraphs = pd.DataFrame(rows)

# Save to CSV
df_paragraphs.to_csv("output/text/paragraph_sentiment_with_metadata.csv", index=False, encoding="utf-8")

print("Created df_paragraphs with shape:", df_paragraphs.shape)
print("Columns:", df_paragraphs.columns.tolist())


Created df_paragraphs with shape: (3106, 18)
Columns: ['body', 'source_file', 'title', 'newspaper', 'date', 'section', 'word_count', 'load_date', 'date_str_clean', 'date_dt', 'geo_hits', 'language', 'article_index', 'paragraph_id', 'paragraph_text', 'sentiment', 'sentiment_stars', 'sentiment_score']


In [ ]:
# location_matching.py

import re
from dataclasses import dataclass
from typing import List, Optional, Tuple, Dict

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

import spacy
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter


# -----------------------------
# 1. NER focused on place mentions
# -----------------------------
nlp = spacy.load(
    "nl_core_news_lg",
    disable=["tagger", "parser", "lemmatizer", "attribute_ruler"],
)
PLACE_LABELS = {"GPE", "LOC"}  # geopolitical entity, location


def extract_place_strings(text: str) -> List[str]:
    """Extract unique place strings from text using spaCy NER."""
    if not text or not str(text).strip():
        return []

    doc = nlp(str(text))
    places: List[str] = []

    for ent in doc.ents:
        if ent.label_ in PLACE_LABELS:
            s = ent.text.strip()
            s = re.sub(r"^[\W_]+|[\W_]+$", "", s)  # trim punctuation
            if len(s) >= 2:
                places.append(s)

    # keep order, de-duplicate (case-insensitive)
    seen = set()
    out = []
    for p in places:
        key = p.lower()
        if key not in seen:
            seen.add(key)
            out.append(p)

    return out


# -----------------------------
# 2. Geocoding (online) with caching + rate limiting
# -----------------------------
geolocator = Nominatim(user_agent="geo-sentiment-mapper")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.0, swallow_exceptions=True)


@dataclass(frozen=True)
class GeoCandidate:
    query: str
    name: str
    lat: float
    lon: float
    importance: float
    point: Point


_geocode_cache: Dict[str, List[GeoCandidate]] = {}


def geocode_place(place: str, country_codes: Optional[str] = None, limit: int = 5) -> List[GeoCandidate]:
    """Return a list of candidate points for a place name."""
    place = str(place).strip()
    if not place:
        return []

    key = f"{place}||{country_codes or ''}||{limit}"
    if key in _geocode_cache:
        return _geocode_cache[key]

    results: List[GeoCandidate] = []
    locs = geocode(place, exactly_one=False, addressdetails=False, country_codes=country_codes, limit=limit)

    if locs:
        for loc in locs:
            raw = getattr(loc, "raw", {}) or {}
            imp = float(raw.get("importance", 0.0) or 0.0)
            results.append(
                GeoCandidate(
                    query=place,
                    name=str(loc),
                    lat=float(loc.latitude),
                    lon=float(loc.longitude),
                    importance=imp,
                    point=Point(float(loc.longitude), float(loc.latitude)),
                )
            )

    _geocode_cache[key] = results
    return results


# -----------------------------
# 3. Spatial filter
# -----------------------------
def candidates_within_polygon(cands: List[GeoCandidate], poly) -> List[GeoCandidate]:
    """Keep only candidates whose point is inside the polygon (or multipolygon)."""
    if not cands or poly is None:
        return []
    return [c for c in cands if poly.contains(c.point)]


# -----------------------------
# 4. Choose ONE best location per paragraph
# -----------------------------
def choose_best_location(
    paragraph_text: str,
    place_strings: List[str],
    poly,
    country_codes: Optional[str] = None,
) -> Tuple[Optional[GeoCandidate], float]:
    """
    Returns (best_candidate, best_score). If no valid candidate inside polygon, returns (None, 0).
    Heuristic scoring:
      - mention frequency in paragraph (more mentions => higher)
      - earlier mention (earlier => higher)
      - geocoder importance (higher => higher)
    """
    if not place_strings or poly is None:
        return None, 0.0

    text = str(paragraph_text)
    text_lower = text.lower()

    best: Optional[GeoCandidate] = None
    best_score = 0.0

    for p in place_strings:
        p_l = p.lower()

        freq = max(1, text_lower.count(p_l))
        pos = text_lower.find(p_l)

        # position bonus: earlier mention gets larger bonus
        if pos >= 0:
            pos_bonus = 1.0 / (1.0 + pos / 100.0)
        else:
            pos_bonus = 0.0

        cands = geocode_place(p, country_codes=country_codes, limit=5)
        cands_in = candidates_within_polygon(cands, poly)
        if not cands_in:
            continue

        # choose best candidate inside polygon
        c_best = max(cands_in, key=lambda c: c.importance)

        score = (2.0 * freq) + (1.0 * pos_bonus) + (1.0 * c_best.importance)

        if score > best_score:
            best_score = score
            best = c_best

    return best, float(best_score)


# -----------------------------
# 5. Apply to df_paragraphs using regions_gdf created earlier
# -----------------------------
def assign_best_locations(
    df_paragraphs: pd.DataFrame,
    regions_gdf: gpd.GeoDataFrame,
    *,
    newspaper_col: str = "newspaper",          # column in df_paragraphs
    text_col: str = "paragraph_text",          # column in df_paragraphs
    regions_newspaper_col: str = "newspaper",  # column in regions_gdf
    country_codes: Optional[str] = "nl,be",    # bias geocoding to NL/BE; set None for global
) -> pd.DataFrame:
    """
    Adds columns:
      - best_place_query
      - best_place_name
      - best_lat
      - best_lon
      - best_loc_score

    Matching logic:
      df_paragraphs[newspaper_col] must match regions_gdf[regions_newspaper_col]
    """

    if newspaper_col not in df_paragraphs.columns:
        raise ValueError(f"df_paragraphs missing column '{newspaper_col}'")

    if text_col not in df_paragraphs.columns:
        raise ValueError(f"df_paragraphs missing column '{text_col}'")

    if regions_newspaper_col not in regions_gdf.columns:
        raise ValueError(f"regions_gdf missing column '{regions_newspaper_col}'")

    # Ensure CRS is WGS84 for point-in-polygon tests with lon/lat
    if regions_gdf.crs is None:
        regions_gdf = regions_gdf.set_crs("EPSG:4326")
    else:
        regions_gdf = regions_gdf.to_crs("EPSG:4326")

    # Map newspaper -> polygon
    region_geom = dict(zip(regions_gdf[regions_newspaper_col], regions_gdf["geometry"]))

    out = df_paragraphs.copy()
    out["best_place_query"] = None
    out["best_place_name"] = None
    out["best_lat"] = pd.NA
    out["best_lon"] = pd.NA
    out["best_loc_score"] = 0.0

    for i, row in out.iterrows():
        paper = row.get(newspaper_col)
        poly = region_geom.get(paper, None)
        text = row.get(text_col, "")

        if poly is None or not text:
            continue

        places = extract_place_strings(str(text))
        best, score = choose_best_location(str(text), places, poly, country_codes=country_codes)

        if best is not None:
            out.at[i, "best_place_query"] = best.query
            out.at[i, "best_place_name"] = best.name
            out.at[i, "best_lat"] = best.lat
            out.at[i, "best_lon"] = best.lon
            out.at[i, "best_loc_score"] = score

    return out


In [ ]:
import pandas as pd

NATIONAL_KEY = "NL_NATIONAL"   # must exist in regions_gdf["newspaper"]

def normalize_newspaper_value(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return pd.NA

    # Your case: values are sets
    if isinstance(x, set):
        if len(x) == 0:
            return pd.NA
        if len(x) == 1:
            return next(iter(x))  # unwrap single newspaper
        return NATIONAL_KEY       # multiple newspapers -> national

    # If anything else slips through, coerce to string
    return str(x).strip() or pd.NA


df_paragraphs["newspaper_norm"] = df_paragraphs["newspaper"].apply(normalize_newspaper_value)

# Optional: inspect what changed
print(df_paragraphs["newspaper_norm"].value_counts().head(20))


newspaper_norm
AD/Haagsche Courant              448
NL_NATIONAL                      305
Trouw                            224
De Twentsche Courant Tubantia    220
Eindhovens Dagblad               195
de Volkskrant                    188
De Gelderlander                  171
Dagblad van het Noorden          158
Het Parool                       156
De Stentor                       139
BN/DeStem                        123
Brabants Dagblad                 109
AD/Utrechts Nieuwsblad            72
Leeuwarder Courant                66
Noordhollands Dagblad             57
De Limburger                      46
Boerderij Vandaag                 40
Nieuwe Oogst                      36
AD/Rotterdams Dagblad             31
AD/Groene Hart                    23
Name: count, dtype: int64


In [ ]:
df_with_locs = assign_best_locations(
    df_paragraphs=df_paragraphs,
    regions_gdf=regions_gdf,
    newspaper_col="newspaper_norm",
    text_col="paragraph_text",
    country_codes="nl,be",
)

df_with_locs.to_csv("output/text/paragraphs_with_best_location.csv", index=False, encoding="utf-8")


RateLimiter caught an error, retrying (0/2 tries). Called with (*('stad Utrecht',), **{'exactly_one': False, 'addressdetails': False, 'country_codes': 'nl,be', 'limit': 5}).
Traceback (most recent call last):
  File "/Users/Yannick/anaconda3/envs/bertopic311/lib/python3.11/site-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "/Users/Yannick/anaconda3/envs/bertopic311/lib/python3.11/site-packages/urllib3/connection.py", line 565, in getresponse
    httplib_response = super().getresponse()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "/Users/Yannick/anaconda3/envs/bertopic311/lib/python3.11/http/client.py", line 1395, in getresponse
    response.begin()
  File "/Users/Yannick/anaconda3/envs/bertopic311/lib/python3.11/http/client.py", line 325, in begin
    version, status, reason = self._read_status()
                              ^^^^^^^^^^^^^^^^^^^
  File "/Users/Yannick/anaconda3/envs/ber

In [ ]:
df_filtered = df_with_locs[df_with_locs["best_place_query"].notna()]

print("Paragraphs with assigned best location:", len(df_filtered))


Paragraphs with assigned best location: 1142


In [ ]:
import geopandas as gpd

# 1) make a GeoDataFrame from best_lon/best_lat
gdf_points = gpd.GeoDataFrame(
    df_filtered.copy(),
    geometry=gpd.points_from_xy(df_filtered["best_lon"], df_filtered["best_lat"]),
    crs="EPSG:4326",
)

# Optional: drop rows with no location chosen
gdf_points = gdf_points.dropna(subset=["best_lon", "best_lat"])

# 2) write outputs
regions_gdf.to_file("output/text/geothermal_locations.gpkg", layer="newspaper_regions", driver="GPKG")
gdf_points.to_file("output/text/geothermal_locations.gpkg", layer="paragraph_points", driver="GPKG")


In [ ]:
import numpy as np
import folium
from folium.plugins import HeatMap

# Ensure numeric
df = df_filtered.copy()
df["best_lat"] = pd.to_numeric(df["best_lat"], errors="coerce")
df["best_lon"] = pd.to_numeric(df["best_lon"], errors="coerce")

# Drop missing coords
df = df.dropna(subset=["best_lat", "best_lon", "sentiment"])

# Split
pos = df[df["sentiment"] == 2]
neg = df[df["sentiment"] == 1]

# Center map
center = [df["best_lat"].mean(), df["best_lon"].mean()]
m = folium.Map(location=center, zoom_start=4, tiles="CartoDB positron")

# Heatmap data: [lat, lon, weight]
pos_data = pos[["best_lat", "best_lon"]].assign(w=1.0).values.tolist()
neg_data = neg[["best_lat", "best_lon"]].assign(w=1.0).values.tolist()

# Custom gradients (tweak as you like)
pos_gradient = {0.0: "transparent", 0.4: "#a1d99b", 0.7: "#31a354", 1.0: "#006d2c"}
neg_gradient = {0.0: "transparent", 0.4: "#fcae91", 0.7: "#fb6a4a", 1.0: "#cb181d"}

HeatMap(
    pos_data,
    name="Positive sentiment",
    radius=18,
    blur=22,
    max_zoom=6,
    gradient=pos_gradient,
).add_to(m)

HeatMap(
    neg_data,
    name="Negative sentiment",
    radius=18,
    blur=22,
    max_zoom=6,
    gradient=neg_gradient,
).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

# in Jupyter this renders; otherwise: m.save("sentiment_map.html")

m.save("sentiment_map.html")

In [10]:
import os
import json
import time
import hashlib
import requests
import pandas as pd
from tqdm.auto import tqdm
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "llama3.1:8b"

SYSTEM = (
    "You are an assistant that extracts the single primary geographic location "
    "a newspaper paragraph is mainly about."
)

def _fingerprint(text: str, region: str) -> str:
    h = hashlib.sha256()
    h.update((region or "").encode("utf-8"))
    h.update(b"\n")
    h.update((text or "").encode("utf-8"))
    return h.hexdigest()

@retry(
    reraise=True,
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=1, max=10),
    retry=retry_if_exception_type((requests.Timeout, requests.ConnectionError, requests.HTTPError)),
)
def llm_primary_location(text: str, newspaper_region_name: str = "Nederland") -> dict:
    prompt = f"""
Task: Determine the ONE primary geographic location this paragraph is mainly about.

Context:
- The newspaper's coverage region is: {newspaper_region_name}
- The paragraph is about geothermal energy.

Rules:
- Return exactly ONE location name, or "NONE" if no clear primary location.
- Prefer the most specific location that is clearly the focus (site/city/municipality).
- If the paragraph is general or national-level, return "Nederland".
- Do NOT list multiple places.
- Output MUST be valid JSON only, with keys:
  location, granularity, confidence, reasoning_short

Granularity must be one of: site, city, municipality, province, country, none
Confidence must be a number from 0 to 1.

Paragraph:
{text}
""".strip()

    payload = {
        "model": MODEL,
        "system": SYSTEM,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.0, "num_predict": 200},
    }

    r = requests.post(OLLAMA_URL, json=payload, timeout=120)
    r.raise_for_status()
    out = (r.json().get("response") or "").strip()

    # robust JSON parsing
    start = out.find("{")
    end = out.rfind("}")
    if start == -1 or end == -1:
        return {"location": "NONE", "granularity": "none", "confidence": 0.0, "reasoning_short": "No JSON returned."}

    try:
        obj = json.loads(out[start:end + 1])
    except json.JSONDecodeError:
        return {"location": "NONE", "granularity": "none", "confidence": 0.0, "reasoning_short": "Invalid JSON."}

    # normalize / clamp
    loc = obj.get("location", "NONE") or "NONE"
    gran = obj.get("granularity", "none") or "none"
    conf = obj.get("confidence", 0.0) or 0.0
    try:
        conf = float(conf)
    except Exception:
        conf = 0.0
    conf = max(0.0, min(1.0, conf))

    reason = obj.get("reasoning_short", "") or ""
    return {"location": loc, "granularity": gran, "confidence": conf, "reasoning_short": reason}


def batch_primary_locations_resumable(
    df: pd.DataFrame,
    text_col: str = "paragraph_text",
    region_col: str = "region_name",
    checkpoint_path: str = "llm_locations_checkpoint.parquet",
    cache_path: str = "llm_locations_cache.jsonl",
    save_every: int = 25,
    sleep_s: float = 0.0,
):
    """
    - Resumes from checkpoint_path if it exists.
    - Writes progress checkpoints every save_every rows (and on Ctrl+C).
    - Uses a small on-disk JSONL cache keyed by hash(region+text) to avoid repeats.
    """

    # --- load checkpoint if present ---
    if os.path.exists(checkpoint_path):
        out = pd.read_parquet(checkpoint_path)
        # If the incoming df has extra rows, align/merge by index
        # (If you have a stable id column, prefer merging on that!)
        out = out.reindex(df.index)
        # keep original cols from df if missing in checkpoint
        for c in df.columns:
            if c not in out.columns:
                out[c] = df[c]
    else:
        out = df.copy()
        out["llm_location"] = None
        out["llm_granularity"] = None
        out["llm_confidence"] = 0.0
        out["llm_reasoning_short"] = None
        out["llm_status"] = None          # ok / empty / error
        out["llm_error"] = None

    # --- load cache (hash -> result dict) ---
    cache = {}
    if os.path.exists(cache_path):
        with open(cache_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                    cache[obj["key"]] = obj["value"]
                except Exception:
                    pass

    def cache_get(key: str):
        return cache.get(key)

    def cache_put(key: str, value: dict):
        if key in cache:
            return
        cache[key] = value
        with open(cache_path, "a", encoding="utf-8") as f:
            f.write(json.dumps({"key": key, "value": value}, ensure_ascii=False) + "\n")

    # --- decide what to process (only missing) ---
    def is_done(row) -> bool:
        # treat any non-null status 'ok' or 'empty' as done; rerun 'error'
        st = row.get("llm_status")
        return st in ("ok", "empty")

    todo_idx = [i for i, r in out.iterrows() if not is_done(r)]

    pbar = tqdm(todo_idx, desc="Geothermal location extraction", unit="row")
    ok = int((out.get("llm_status") == "ok").sum()) if "llm_status" in out.columns else 0
    err = int((out.get("llm_status") == "error").sum()) if "llm_status" in out.columns else 0

    processed_since_save = 0

    try:
        for i in pbar:
            text = str(out.at[i, text_col] if text_col in out.columns else "") or ""
            region_name = str(out.at[i, region_col] if region_col in out.columns else "Nederland") or "Nederland"

            if not text.strip():
                out.at[i, "llm_status"] = "empty"
                out.at[i, "llm_location"] = "NONE"
                out.at[i, "llm_granularity"] = "none"
                out.at[i, "llm_confidence"] = 0.0
                out.at[i, "llm_reasoning_short"] = "Empty paragraph."
                processed_since_save += 1
                continue

            key = _fingerprint(text, region_name)
            cached = cache_get(key)

            try:
                res = cached if cached is not None else llm_primary_location(text, newspaper_region_name=region_name)
                if cached is None:
                    cache_put(key, res)

                out.at[i, "llm_location"] = res.get("location")
                out.at[i, "llm_granularity"] = res.get("granularity")
                out.at[i, "llm_confidence"] = float(res.get("confidence") or 0.0)
                out.at[i, "llm_reasoning_short"] = res.get("reasoning_short")
                out.at[i, "llm_status"] = "ok"
                out.at[i, "llm_error"] = None
                ok += 1
            except Exception as e:
                out.at[i, "llm_status"] = "error"
                out.at[i, "llm_error"] = repr(e)
                err += 1

            processed_since_save += 1
            pbar.set_postfix({"ok": ok, "error": err, "cached": cached is not None})

            if sleep_s:
                time.sleep(sleep_s)

            if processed_since_save >= save_every:
                out.to_parquet(checkpoint_path, index=True)
                processed_since_save = 0

    except KeyboardInterrupt:
        # Graceful stop: save checkpoint, then re-raise to exit loop if you want
        out.to_parquet(checkpoint_path, index=True)
        print(f"\nStopped by user. Progress saved to: {checkpoint_path}")
        return out

    # final save
    out.to_parquet(checkpoint_path, index=True)
    return out


In [11]:
import hashlib

def make_uid(row) -> str:
    base = "||".join([
        str(row.get("source", "")),
        str(row.get("document_title", "")),
        str(row.get("publish_date", "")),
        str(row.get("paragraph_id", "")),
        str(row.get("paragraph_text", "")),
    ])
    return hashlib.sha256(base.encode("utf-8")).hexdigest()

df["uid"] = df.apply(make_uid, axis=1)
df = df.set_index("uid", drop=False)


In [ ]:
out = batch_primary_locations_resumable(
    df,
    text_col="paragraph_text",
    region_col="region_name",
    checkpoint_path="geo_checkpoint.parquet",
    cache_path="geo_cache.jsonl",
    save_every=25,
    sleep_s=0.0,
)

out.to_csv("output/text/llm_locations.csv", index=False, encoding="utf-8")


ValueError: cannot reindex on an axis with duplicate labels

In [9]:
import pandas as pd
import os

os.chdir("/Users/Yannick/dev/geothermal_text_mining")

df = pd.read_csv("output/italian/text/articles_cleaned.csv", encoding="utf-8")

In [10]:
print(df.head())

df.head(2).to_csv("output/italian/text/sample_2.csv", index=False, encoding="utf-8")

    source_file                                        source_path  \
0  italian1.RTF  /Users/Yannick/dev/geothermal_text_mining/inpu...   
1  italian1.RTF  /Users/Yannick/dev/geothermal_text_mining/inpu...   
2  italian8.RTF  /Users/Yannick/dev/geothermal_text_mining/inpu...   
3  italian8.RTF  /Users/Yannick/dev/geothermal_text_mining/inpu...   
4  italian8.RTF  /Users/Yannick/dev/geothermal_text_mining/inpu...   

                                               title  \
0                                       User Name: =   
1  Le analisi nelle carte dell'accusa “Arsenico 2...   
2  Controllo sismico Accordo con Unife; La decisi...   
3  Sembrano mattoni, ma è legno A Ravenna la prim...   
4  “Falde acquifere, la presenza di tallio è sott...   

                                           newspaper            date  \
0  36. Cecina, Fofana in prestito al Rosignano e ...     7 July 2015   
1                       Il Resto del Carlino (Italy)     7 July 2015   
2                       Il

In [11]:
print(df["newspaper"].value_counts())

newspaper
La Nazione (Italy)                                                                                                                                                              2229
Il Resto del Carlino (Italy)                                                                                                                                                     303
Corriere della Sera (Italy)                                                                                                                                                      241
Il Giorno (Italy)                                                                                                                                                                120
MF                                                                                                                                                                                43
ItaliaOggi                                                                           